## <font color="royalblue">**Extracción de Datos**</font>
### <font color="royalblue">**Web Scraping de LinkedIn**</font>
En esta archivo se definen todos los pasos para extraer del portal web LinkedIn ofertas de empleo en Data Analyst para Barcelona.
Un proceso de web scraping involucra acceder a páginas web que poseen medios de protección, por tal motivo el ingreso se realizo con 2 herramientas; el uso de selenium.webdriver y logeo con cuenta a través de cookies.
- Se cargan las librerias necesarias para el proceso (Selenium, BeautifulSoap, requests).  

**Paso 1. Acceso al portal web**
- Las funciones *guardar_cookies*, *iniciar_sesion_con_cookies*; nos permiten almacenar las cookies de una cuenta e iniciar sesión con la misma.   

**Paso 2. Extraccción de datos** 
- Las funciones *ir_a_busqueda*, *scroll_infinito_linkedin*, *extraer_id*, *extraer_jocards*, *scrapear_paginas*: permiten ir a LinkedIn jobs, cargar las ofertas de empleo e iniciar la extraccion de datos.  

**Paso 3. Limpieza y extracción de descripción de oferta**
- Las funciones *limpiar_ofertas*, *obtener_job_id*, *construir_url_job*, *obtener_fecha_publicacion* , *obtener_detalles_selenium*, *agregar_descripciones_selenium*: permiten acondicionar las ofertas extraidas y en el caso de las descripciones que implica acceder a cada id y url de forma individual se hizo en un proceso con Selenium.

- Finalmente se creo un dataframe *df_LinkedIn* para visualizar los datos y fueron exportados en archivo .csv para posterior preprocesamiento.

  

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time
import json
import re
import requests
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd


In [ ]:
# Función para guardar cookies de LinkedIn (SOLO SE EJECUTA UNA VEZ)
def guardar_cookies():
    driver = webdriver.Chrome()
    driver.get("https://www.linkedin.com/login")
    
    print("👉 Inicia sesión manualmente con Google o LinkedIn.")
    input("Presiona ENTER cuando hayas iniciado sesión...")

    cookies = driver.get_cookies()
    with open("linkedin_cookies.json", "w") as f:
        json.dump(cookies, f)

    print("Cookies guardadas correctamente.")
    driver.quit()

guardar_cookies()


👉 Inicia sesión manualmente con Google o LinkedIn.
Cookies guardadas correctamente.


In [ ]:
# Función para iniciar en chrome con cookies
def iniciar_sesion_con_cookies():
    print(">>> Ejecutando iniciar_sesion_con_cookies()")

    driver = webdriver.Chrome()

    # 1. Cargar LinkedIn para inicializar dominio
    driver.get("https://www.linkedin.com")
    time.sleep(2)

    # 2. Cargar cookies
    try:
        with open("linkedin_cookies.json", "r") as f:
            cookies = json.load(f)
    except FileNotFoundError:
        print("❌ No se encontró linkedin_cookies.json")
        return

    for cookie in cookies:
        try:
            driver.add_cookie(cookie)
        except Exception as e:
            pass

    # 3. Recargar LinkedIn ya con cookies
    driver.get("https://www.linkedin.com/feed/")
    time.sleep(3)

    print("✔ Sesión restaurada con cookies.")
    return driver


In [3]:
# Ejecucion de prueba en chrome con cookies
driver = iniciar_sesion_con_cookies()

>>> Ejecutando iniciar_sesion_con_cookies()
✔ Sesión restaurada con cookies.


In [ ]:
# Función para navegar a LinkedIn Jobs
def ir_a_busqueda(driver, query, location):
    url = f"https://www.linkedin.com/jobs/search/?keywords={query}&location={location}&refresh=true"
    driver.get(url)
    time.sleep(4)

# Función para hacer scroll infinito y cargar más resultados
def scroll_infinito_linkedin(driver, ciclos=60):
    for i in range(ciclos):
        try:
            boton = driver.find_element(
                By.CLASS_NAME, "infinite-scroller__show-more-button"
            )
            driver.execute_script("arguments[0].click();", boton)
            time.sleep(1.2)
        except:
            print("✔ No hay más resultados para cargar.")
            break


In [ ]:
# Función para extraer id de las ofertas
def extraer_id(url):
    match = re.search(r'/jobs/view/(\d+)', url)
    return match.group(1) if match else None

# Función para extraer información de las ofertas
def extraer_jobcards(driver):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.job-card-container--clickable")
    ofertas = []

    for card in cards:
        datos = {}

        # Título
        try:
            datos["titulo"] = card.find_element(By.CSS_SELECTOR, "div.artdeco-entity-lockup__title").text.strip()
        except:
            datos["titulo"] = None

        # Empresa
        try:
            datos["empresa"] = card.find_element(By.CSS_SELECTOR, "div.artdeco-entity-lockup__subtitle").text.strip()
        except:
            datos["empresa"] = None

        # Ubicación / modalidad / contrato
        try:
            datos["ubicacion"] = card.find_element(By.CSS_SELECTOR, "div.artdeco-entity-lockup__caption").text.strip()
        except:
            datos["ubicacion"] = None

        # URL
        try:
            datos["url"] = card.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
        except:
            datos["url"] = None

        # ID
        datos["id"] = extraer_id(datos["url"]) if datos["url"] else None

        # Fecha
        try:
            datos["fecha"] = card.find_element(By.CSS_SELECTOR, "time").text.strip()
        except:
            datos["fecha"] = None

        ofertas.append(datos)

    print(f"Extraídas {len(ofertas)} ofertas.")
    return ofertas
# usa extraer_id


In [ ]:
# Función principal para scrapeo de oferta en linkedin
def scrapear_paginas(driver, query, location, num_paginas=4):
    ir_a_busqueda(driver, query, location)
    time.sleep(3)

    ofertas_totales = []

    pagina = 1
    while pagina <= num_paginas:
        print(f"📄 Procesando página {pagina}...")

        # Scroll infinito en la página actual
        scroll_infinito_linkedin(driver, ciclos=60)

        # Extraer ofertas de esta página
        ofertas = extraer_jobcards(driver)
        ofertas_totales.extend(ofertas)

        # Intentar encontrar el botón "Siguiente"
        try:
            boton_siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "button.jobs-search-pagination__button--next"
            )

            # Si está deshabilitado, no hay más páginas
            if not boton_siguiente.is_enabled():
                print("✔ No hay más páginas disponibles.")
                break

            boton_siguiente.click()
            time.sleep(3)
            pagina += 1

        except:
            print("✔ No se encontró botón 'Siguiente'. Fin de paginación.")
            break

    return ofertas_totales
# usa ir_a_busqueda, scroll_infinito_linkedin, extraer_jobcards. 


In [ ]:
# Función para limpiar ofertas 
def limpiar_ofertas(ofertas):
    urls_vistas = set()
    ofertas_limpias = []

    for oferta in ofertas:
        url = oferta.get("url")

        # Saltar ofertas sin URL
        if not url:
            continue

        # Normalizar URL (quitar parámetros)
        url = url.split("?")[0]
        oferta["url"] = url

        # Evitar duplicados
        if url in urls_vistas:
            continue

        urls_vistas.add(url)
        ofertas_limpias.append(oferta)

    print(f"✔ Ofertas limpias: {len(ofertas_limpias)}")
    return ofertas_limpias


In [ ]:
# Funciones para extraer descripción completa  →  Web Scraping Selenium
def obtener_job_id(url):
    try:
        return url.split("/view/")[1].split("/")[0]
    except:
        return None

def construir_url_job(job_id):
    return f"https://www.linkedin.com/jobs/search/?currentJobId={job_id}"

def obtener_fecha_publicacion(driver):
    try:
        fecha_elem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="workspace"]/div/div/div[1]/div/div/div/div[1]/div/div/div[2]/div/div[1]/p/span[4]')
            )
        )
        return fecha_elem.text.strip()
    except:
        return None

def obtener_detalles_selenium(driver, url):
    job_id = obtener_job_id(url)
    if not job_id:
        return None, None

    url_correcta = construir_url_job(job_id)
    driver.get(url_correcta)

    # 1. Extraer descripción
    descripcion = None
    selectores = [
        "div.jobs-box__html-content",
        "div.show-more-less-html__markup"
    ]

    for selector in selectores:
        try:
            elem = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )
            descripcion = elem.text.strip()
            break
        except:
            pass

    # 2. Extraer fecha
    fecha = obtener_fecha_publicacion(driver)

    return descripcion, fecha
# usa obtener_job_id, construir_url_job, obtener_fecha_publicacion.

def agregar_descripciones_selenium(driver, ofertas):
    for i, oferta in enumerate(ofertas, start=1):
        print(f"Procesando oferta {i}/{len(ofertas)}")

        descripcion, fecha = obtener_detalles_selenium(driver, oferta["url"])

        oferta["descripcion"] = descripcion
        oferta["fecha_publicacion"] = fecha

    return ofertas
# usa obtener_detalles_selenium.


In [ ]:
# Pipeline completo, extraccion de descripcion con Selenium
driver = iniciar_sesion_con_cookies()

ofertas = scrapear_paginas(
    driver,
    "data analyst",
    "Barcelona",
    num_paginas = 1   # ← aquí controlas cuántas páginas quieres
)

ofertas = limpiar_ofertas(ofertas)
ofertas = agregar_descripciones_selenium(driver, ofertas)

driver.quit()


>>> Ejecutando iniciar_sesion_con_cookies()
✔ Sesión restaurada con cookies.
📄 Procesando página 1...
✔ No hay más resultados para cargar.
Extraídas 7 ofertas.
✔ Ofertas limpias: 7
Procesando oferta 1/7
Procesando oferta 2/7
Procesando oferta 3/7
Procesando oferta 4/7
Procesando oferta 5/7
Procesando oferta 6/7
Procesando oferta 7/7


In [ ]:
# Guardar en un DataFrame resultados de ofertas
df_Linkedin = pd.DataFrame(ofertas)
df_Linkedin.head()

,titulo,empresa,ubicacion,url,id,fecha,descripcion,fecha_publicacion
0,Control de Calidad de Datos (IA e Inspección d...,SEWDEF,"Barcelona, Cataluña, España (Híbrido)",https://www.linkedin.com/jobs/view/4399418207/,4399418207,None,Acerca del empleo\nSobre el rol\nTu objetivo s...,None
1,Coordinadores/as dTècnic/a d’Investigació Sèni...,Sigma Dos,"Barcelona, Cataluña, España (Presencial)",https://www.linkedin.com/jobs/view/4387878667/,4387878667,None,Acerca del empleo\n\nOPORTUNITAT LABORAL: Tècn...,None
2,Junior Data Analyst\nJunior Data Analyst with ...,Evolve,España (En remoto),https://www.linkedin.com/jobs/view/4402262919/,4402262919,None,Acerca del empleo\nDescripción de la oferta\nE...,None
3,Business Data Analyst\nBusiness Data Analyst,Volkswagen Group Services Barcelona,"Martorell, Cataluña, España (Híbrido)",https://www.linkedin.com/jobs/view/4397282199/,4397282199,None,Acerca del empleo\nEn Volkswagen Group Service...,None
4,📍 Data Scientist – ByRatings (Remote-Friendly ...,ByRatings,"Barcelona, Cataluña, España (Híbrido)",https://www.linkedin.com/jobs/view/4368494721/,4368494721,None,Acerca del empleo\n¿Quiénes somos?\n\nEn ByRat...,None


In [ ]:
# Exportar resultados a CSV del Web Scraping al portal de LinkedIn
df_Linkedin.to_csv("Linkedin.csv", index=False)